youtube video for github link with databricks

https://www.youtube.com/watch?v=uljYTx7nzy8

In [0]:
print("Hello Databricks!")

### Spark Architecture

![image_1776509406690.png](./image_1776509406690.png "image_1776509406690.png") [youtube](https://www.youtube.com/watch?v=nKRwjXRs4mY)

When we run code in Databricks, the Driver node first analyzes the work. It asks the Cluster Manager to identify available resources (Workers) to handle the job. The Cluster Manager assigns the work, and the Driver distributes specific tasks to the Workers to be processed in parallel.

Once the Workers finish, they send their results back to the Driver, which combines them into a final output for the client.

* **Worker Node:** The physical machine or "building" that provides CPU and RAM.
* **Executor:** The actual software process or "worker" living inside that machine that runs your code.

An important note:

* Driver = brain (decides work)
* Executor = worker (does work)
* Task = small job(each worker has multiple jobs)
* Core = how many tasks a worker can do at once(if worker has 3 hands, he/she can do 3 tasks at a time)

Note: Spark uses 'Lazy Evaluation.' It won't actually execute any data processing until an 'Action' (like show(), count(), or display()) is called. Until then, it simply keeps a plan of what needs to be done.

[youtube](https://www.youtube.com/watch?v=NgQP08BsxkI&list=PLlHuhl-cAAnEgsGlq2H_iV50jUhiFayN_&index=4)

If we use databricks for pyspark, the platforma handles everything ![image_1776507747971.png](./image_1776507747971.png "image_1776507747971.png")

In [0]:
from pyspark.sql import SparkSession

#`SparkSession` is the entry point to programming with Spark.
# we use it to create DataFrames, read data, and execute queries.
spark = SparkSession.builder.appName('SparkBasics').getOrCreate()

SQLContext is used for basic SQL operations on DataFrames, while HiveContext extends it with Hive(SQL on Big Data) support like Hive tables and metastore. In modern Spark, both are replaced by SparkSession.

## Creating DataFrames in Different Ways

In [0]:
# From a List of Rows
data = [
  ('Alice', 25), ('Bob', 23), ('Charlie', 35)
]

columns = ['name', 'age']

df = spark.createDataFrame(data, schema = columns)

display(df)

In [0]:
# Using a Dictionary (Pythonic Way)
data = [
    {'Name': 'Alice', 'Age': 25},
    {'Name': 'Bob', 'Age': 23},
    {'Name': 'Charlie', 'Age': 35}
]

df = spark.createDataFrame(data)

display(df)

# Wrong way
data = [
    {'Name': ['Alice', 'Bob', 'Charlie'], 'Age': [25, 23, 35]}
]
display(spark.createDataFrame(data))

In [0]:
# Reading the sample table file in databricks(in the catalog we have 'samples' there we get sample files.)

# youtube: https://www.youtube.com/watch?v=FXwqc4zJBhw&list=PLlHuhl-cAAnEgsGlq2H_iV50jUhiFayN_&index=6

sample_df = spark.read.table('samples.bakehouse.sales_suppliers')
display(sample_df.limit(10))

### Using the Catalog to upload the data

**How to add the data**

Go to the Catalog, click 'Add data', then 'Create or modify table', there we can see the path(catalog->workspace->default) where our table will be created.

In [0]:
bms_df = spark.read.table('workspace.default.big_mart_sales')
display(bms_df.limit(5))

In [0]:
# Use the table name directly, no paths required
df = spark.table("workspace.default.ecommerce_sales_analysis")

display(df)

In PySpark, DataFrames are immutable. This means we cannot "edit" or change an existing DataFrame in place. Instead, we perform an operation that creates a new DataFrame with the changes you want.

In [0]:
# Updating the 'Order ID' column
from pyspark.sql.functions import regexp_replace

df1 = df.withColumn('OrderID', regexp_replace('Order ID', 'CA-', ''))
display(df1.limit(10))

In [0]:
from pyspark.sql.functions import col, when
df1 = df.withColumn('Ship_Mode', when(col('Ship Mode') == 'Second Class', '2nd class')
                                .when(col('Ship Mode') == 'Standard Class', 'Standard')
                                .when(col("Ship Mode") == 'First Class', '1st class')
                                .otherwise(col('Ship Mode'))
                    )

display(df1.limit(10))

## Unity Catalog - "Central Admin" for a company's data.

Unity Catalog is a single place to manage all data and permissions. Instead of having data scattered everywhere with different security rules for each team, Unity Catalog brings it all under one 'roof'—using a simple Catalog > Schema > Table structure—so we can easily control who sees what, no matter how big the company gets.

1. Unified
    * What it means: Bringing things together that were previously separated.
2. Governance
    * What it means: The rules that decide who is allowed to do what with the data.
3. Centralized 'Map'
    * What it means: A single place to find where all data is located.
4. 'Security Guard'
    * What it means: A system that stops unauthorized people from touching things.

So, I think of Unity Catalog as the 'Brain' of the company's data. 
* It's Unified because it gathers everything into one place.
* It acts as a Centralized Map so we always know where our data is.
* It provides Governance by letting us set clear rules.
* And it acts as a Security Guard to make sure only the right people are allowed to touch that data.

## Volume

In [0]:
## Creating a directory
dbutils.fs.mkdirs("/Volumes/workspace/default/my_first_vol/input_files")

In [0]:
path = '/Volumes/workspace/default/my_first_vol/insurance.csv'

df = spark.read.format("csv")\
    .option("header", "true")\
    .load(path)

display(df)